# Banking Customer Churn Prediction

## Problem Statement

A retail bank wants to identify customers who are likely to leave the bank.

The objective is to predict **Churn (Yes/No)** using customer demographics, account information, financial profile, banking-channel usage, transaction activity, complaints, and product ownership.

The model can help the bank identify high-risk customers and support proactive customer-retention campaigns.

In [ ]:
## Import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

import joblib

In [ ]:
# Load dataset

DATA_PATH = 'banking_customer_churn_50000.csv'

df = pd.read_csv(DATA_PATH)

In [ ]:
# Display dataset

df

In [ ]:
## Basic data check

df.info()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe(include='all').T

In [ ]:
## Check duplicate records

print('Duplicate rows:', df.duplicated().sum())

In [ ]:
## Remove duplicate records

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print('Shape after removing duplicates:', df.shape)

### Data quality check

The generated banking dataset intentionally contains:

- Missing values
- Error values such as `?`, `@`, `#`, `%`
- Outliers
- Duplicate records

These are handled before model training.

In [ ]:
## Check missing values

missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

In [ ]:
## Identify error values

error_values = ['?', '@', '#', '%']

for col in df.select_dtypes(include='object').columns:

    error_count = df[col].isin(error_values).sum()

    if error_count > 0:
        print(f'{col}: {error_count} error values')

In [ ]:
## Replace error values with NaN

error_values = ['?', '@', '#', '%']

df.replace(
    error_values,
    np.nan,
    inplace=True
)

print('Error values replaced successfully.')

In [ ]:
## Separate categorical and numerical columns

cat = df.select_dtypes(include='object')
num = df.select_dtypes(include=np.number)

print('Categorical columns:')
print(cat.columns.tolist())

print('\nNumerical columns:')
print(num.columns.tolist())

In [ ]:
## Handle missing values

# Categorical columns -> mode
for col in cat.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

# Numerical columns -> median
for col in num.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print('Remaining missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### Outlier handling

For banking financial variables, extreme values can affect distance-based and linear models. We cap numerical outliers using the IQR method instead of blindly deleting customers.

In [ ]:
## Outlier detection and capping

outlier_columns = [
    'Age',
    'Annual_Income',
    'Credit_Score',
    'Employment_Years',
    'Loan_Amount',
    'Monthly_EMI',
    'Debt_to_Income',
    'Account_Balance',
    'Avg_Monthly_Transactions',
    'Cash_Withdrawals',
    'Online_Transactions',
    'Failed_Transactions',
    'Credit_Card_Spend',
    'Customer_Service_Calls',
    'Complaints',
    'Branch_Visits',
    'Last_Transaction_Days'
]

for col in outlier_columns:

    if col in df.columns:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        df[col] = df[col].clip(
            lower=lower_limit,
            upper=upper_limit
        )

print('Outlier treatment completed.')

In [ ]:
## Target distribution

print(df['Churn'].value_counts())

print('\nTarget percentage:')
print(
    df['Churn'].value_counts(normalize=True).mul(100).round(2)
)

In [ ]:
## Target distribution visualization

plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x='Churn'
)

plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Customer Count')

plt.show()

### Feature and target split

`Churn` is the target variable.

`Customer_ID` is removed because it is an identifier and should not provide predictive information.

In [ ]:
## Split the dataset into X and y

TARGET = 'Churn'

X = df.drop(
    columns=[
        'Customer_ID',
        TARGET
    ]
).copy()

y = df[TARGET].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
## Encode target variable

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print('Target classes:')
print(target_encoder.classes_)

In [ ]:
## Encode categorical dataset

categorical_columns = X.select_dtypes(
    include='object'
).columns

label_encoders = {}

for col in categorical_columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    label_encoders[col] = le

print('Encoded categorical columns:')
print(categorical_columns.tolist())

### Train/Test Split

A stratified split is used because churn is a classification problem and the target classes should remain proportionally represented in both training and testing data.

In [ ]:
## Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training shape:', X_train.shape)
print('Testing shape :', X_test.shape)

In [ ]:
## Standardize the dataset

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

print('Feature scaling completed.')

### Multi-model comparison

Compare several standard classification algorithms using stratified cross-validation.

The primary model-selection metric is **F1-score**, because customer churn datasets can have class imbalance and both precision and recall are important.

In [ ]:
## Multiple classification models

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        max_depth=8
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ),

    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42
    ),

    'K-Nearest Neighbors': KNeighborsClassifier()
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = [
    'accuracy',
    'precision',
    'recall',
    'f1',
    'roc_auc'
]

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring=scoring
    )

    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1': scores['test_f1'].mean(),
        'ROC_AUC': scores['test_roc_auc'].mean()
    })

cv_results = pd.DataFrame(
    cv_results
).sort_values(
    by='F1',
    ascending=False
)

print(cv_results.to_string(index=False))

In [ ]:
## Visualize model comparison

cv_results.set_index('Model')[
    ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC']
].plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title('Classification Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=30)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
## Select and train the best model

best_model_name = cv_results.iloc[0]['Model']

print('Best model:', best_model_name)

best_model = models[best_model_name]

best_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
## Predictions

y_pred = best_model.predict(
    X_test_scaled
)

y_probability = best_model.predict_proba(
    X_test_scaled
)[:, 1]

print('Sample predictions:')
print(y_pred[:10])

In [ ]:
## Model evaluation

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print(f'Accuracy  : {accuracy:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'ROC-AUC   : {roc_auc:.4f}')

In [ ]:
## Classification report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
## Confusion matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

cm_df = pd.DataFrame(
    cm,
    index=target_encoder.classes_,
    columns=target_encoder.classes_
)

cm_df

In [ ]:
## Confusion matrix visualization

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm_df,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
## Feature importance

if hasattr(best_model, 'feature_importances_'):

    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values(
        by='Importance',
        ascending=False
    )

    print(
        feature_importance.head(20).to_string(
            index=False
        )
    )

elif hasattr(best_model, 'coef_'):

    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': np.abs(best_model.coef_[0])
    }).sort_values(
        by='Importance',
        ascending=False
    )

    print(
        feature_importance.head(20).to_string(
            index=False
        )
    )

In [ ]:
## Feature importance visualization

if 'feature_importance' in globals():

    top_features = feature_importance.head(15)

    plt.figure(figsize=(10, 7))

    sns.barplot(
        data=top_features,
        x='Importance',
        y='Feature'
    )

    plt.title('Top 15 Important Features')
    plt.tight_layout()

    plt.show()

In [ ]:
## Prediction output

prediction_output = df.iloc[
    X_test.index
].copy()

prediction_output['Actual_Churn'] = target_encoder.inverse_transform(
    y_test
)

prediction_output['Predicted_Churn'] = target_encoder.inverse_transform(
    y_pred
)

prediction_output['Churn_Probability'] = np.round(
    y_probability,
    4
)

prediction_output.head()

In [ ]:
## Save predictions

prediction_output.to_csv(
    'banking_churn_predictions.csv',
    index=False
)

print('Predictions saved successfully.')

In [ ]:
## Save trained model and preprocessing objects

model_package = {
    'model': best_model,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'target_encoder': target_encoder,
    'feature_columns': X.columns.tolist(),
    'target': TARGET
}

joblib.dump(
    model_package,
    'banking_customer_churn_model.pkl'
)

print(
    'Model package saved as banking_customer_churn_model.pkl'
)

### Real-time prediction function

The function below can be reused by an API, batch pipeline, dashboard, or banking application to estimate the probability that a new customer will churn.

In [ ]:
## Real-time prediction function

def predict_churn(new_data):

    data = new_data.copy()

    # Encode categorical features using the encoders
    for col, le in label_encoders.items():

        values = data[col].fillna(
            le.classes_[0]
        ).astype(str)

        mapping = {
            value: index
            for index, value in enumerate(
                le.classes_
            )
        }

        data[col] = values.map(
            mapping
        ).fillna(-1)

    # Keep exactly the training feature order
    data = data[
        X.columns
    ].copy()

    # Standardize
    data_scaled = scaler.transform(
        data
    )

    # Prediction
    prediction = best_model.predict(
        data_scaled
    )

    probability = best_model.predict_proba(
        data_scaled
    )[:, 1]

    result = new_data.copy()

    result['Predicted_Churn'] = (
        target_encoder.inverse_transform(
            prediction
        )
    )

    result['Churn_Probability'] = np.round(
        probability,
        4
    )

    return result

In [ ]:
## Example real-time prediction

sample_customer = df.drop(
    columns=['Churn']
).iloc[[0]].copy()

sample_customer = sample_customer.drop(
    columns=['Customer_ID']
)

predict_churn(
    sample_customer
)

## Final ML Workflow

**Business Problem → Data Understanding → Data Cleaning → Missing Value Handling → Error Value Handling → Outlier Treatment → EDA → Target Analysis → Encoding → Train/Test Split → Feature Scaling → Multi-Model Cross Validation → Best Model Selection → Evaluation → Feature Importance → Prediction → Model Persistence → Real-Time Prediction**

### Business Use Case

The final model can support:

- Customer retention campaigns
- Proactive relationship-manager calls
- Personalized offers
- Digital banking engagement campaigns
- Loan/product cross-selling
- Customer-risk prioritization

### Important ML Engineering Practices

- Customer ID is excluded from model features.
- Error values are converted to missing values.
- Missing numerical values use median imputation.
- Missing categorical values use mode imputation.
- Outliers are capped using the IQR method.
- Stratified train/test splitting preserves churn proportions.
- Scaling is applied after the train/test split.
- Multiple models are compared using cross-validation.
- F1 and ROC-AUC are considered alongside accuracy.
- The complete preprocessing and model pipeline is saved for inference.